In [0]:
%run /Workspace/Repos/Project_1705/Azure_Retail_Project/Project_Files/Functions/ACCESS_ADLS_GEN2_USING_SERVICE_PRINCIPALE

In [0]:
from pyspark.sql.functions import *
from datetime import datetime

In [0]:
# =========================
# Widgets
# =========================

dbutils.widgets.text("read_container", "silver", "Read Container")
dbutils.widgets.text("write_container", "gold", "Write Container")
dbutils.widgets.text("fact_table_name", "fact_orders", "Fact Table Name")

read_container = dbutils.widgets.get("read_container")
write_container = dbutils.widgets.get("write_container")
fact_table_name = dbutils.widgets.get("fact_table_name")

In [0]:

# =========================
# Current Date
# =========================

currentdate = datetime.now().strftime("%Y%m%d")

In [0]:
# =========================
# Base Paths
# =========================

base_read_path = f"abfss://{read_container}@retailstorage1881.dfs.core.windows.net"
base_write_path = f"abfss://{write_container}@retailstorage1881.dfs.core.windows.net"

In [0]:
# =========================
# Generic Read Function
# =========================

def read_delta_table(table_name):
    return spark.read.format("delta").load(
        f"{base_read_path}/{table_name}/{currentdate}"
    )

In [0]:
# =========================
# Read Tables
# =========================

orders_df = read_delta_table("orders")
order_items_df = read_delta_table("order_items")
payments_df = read_delta_table("payments")
reviews_df = read_delta_table("reviews")

In [0]:
# Gold Dimension
dim_date = spark.read.format("delta").load(
    f"{base_write_path}/dim_date"
)

In [0]:
# =========================
# Aggregate Tables
# =========================

payments_agg_df = (
    payments_df
    .groupBy("order_id")
    .agg(sum("payment_value").alias("payment_value"))
)

reviews_agg_df = (
    reviews_df
    .groupBy("order_id")
    .agg(avg("review_score").alias("review_score"))
)

In [0]:

# =========================
# Fact Table Logic
# =========================

fact_df = (
    orders_df.alias("o")

    .join(
        order_items_df.alias("oi"),
        col("o.order_id") == col("oi.order_id"),
        "inner"
    )

    .join(
        payments_agg_df.alias("p"),
        col("o.order_id") == col("p.order_id"),
        "left"
    )

    .join(
        reviews_agg_df.alias("r"),
        col("o.order_id") == col("r.order_id"),
        "left"
    )

    .join(
        dim_date.alias("d"),

        date_format(
            col("o.order_purchase_timestamp"),
            "yyyyMMdd"
        ).cast("int") == col("d.date_key"),

        "left"
    )

    .select(
        col("o.order_id"),
        col("oi.product_id"),
        col("o.customer_id"),
        col("oi.seller_id"),
        col("d.date_key"),
        col("o.order_status"),
        col("oi.price"),
        col("oi.freight_value"),
        col("p.payment_value"),
        col("r.review_score"),

        datediff(
            col("o.order_delivered_customer_date"),
            col("o.order_purchase_timestamp")
        ).alias("delivery_days"),

        current_timestamp().alias("created_date")
    )
)


In [0]:
# =========================
# Display
# =========================

display(fact_df)

In [0]:
# =========================
# Write Gold Fact Table
# =========================

fact_df.write.format("delta") \
    .mode("overwrite") \
    .save(f"{base_write_path}/{fact_table_name}")